# YOLOv8s LSKA / Inner-IoU Long Training on Kaggle

这版 notebook 参考你原来的 `YOLO11s baseline` Kaggle notebook，改成了两个可选实验分支：

- `YOLOv8s + LSKA`
- `YOLOv8s + Inner-IoU`

使用方式：
- 先运行前面的公共准备单元
- 训练时只运行二选一的训练单元
- 训练结束后再运行后面的评估、误差库、打包单元

上传到 Kaggle 前请确认：
- `Settings` 里已经打开 `GPU`
- `Internet` 已打开
- 下面 `DATASET_ROOT` 改成你自己的 Kaggle 数据集路径
- 如果显存不够，优先调小 `BATCH`


In [ ]:
# Cell 1: 克隆代码（直接使用 GitHub 最新版本）
%cd /kaggle/working
!rm -rf yolov8s_kuangjing
!git clone https://github.com/tuanziya666/yolov8s_kuangjing.git
%cd /kaggle/working/yolov8s_kuangjing
!git pull


In [ ]:
# Cell 2: 基础环境检查，并关闭 wandb/raytune
%cd /kaggle/working/yolov8s_kuangjing
!python -c "import torch, cv2, numpy; from ultralytics import YOLO; print('OK'); print('torch=', torch.__version__); print('cv2=', cv2.__version__); print('numpy=', numpy.__version__)"
!python -c "from ultralytics.utils import SETTINGS; SETTINGS.update(wandb=False, raytune=False); print(SETTINGS)"


In [ ]:
# Cell 3: 查看 Kaggle 数据集挂载路径
!find /kaggle/input -maxdepth 3 -type d | head -n 100


In [ ]:
# Cell 4: 设置 Kaggle 数据集路径并写出专用 data.yaml
# 把下面这个路径改成你上传到 Kaggle 的数据集根目录
DATASET_ROOT = "/kaggle/input/your-dataset-root"

from pathlib import Path

yaml_text = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
"""

cfg_path = Path('/kaggle/working/yolov8s_kuangjing/ultralytics/cfg/datasets/coalmine4_kaggle.yaml')
cfg_path.write_text(yaml_text, encoding='utf-8')
print(cfg_path.read_text(encoding='utf-8'))


In [ ]:
# Cell 5: 确认 LSKA 配置、Inner-IoU 训练脚本和数据配置文件
%cd /kaggle/working/yolov8s_kuangjing
!ls -lh ultralytics/cfg/models/v8/yolov8s_lska.yaml
!ls -lh train_yolov8s_inner_iou.py
!cat ultralytics/cfg/datasets/coalmine4_kaggle.yaml


In [ ]:
# Cell 6: 让 DDP 子进程也能找到本地 ultralytics，并补好 AMP 自检图片
%cd /kaggle/working/yolov8s_kuangjing

import os
import sys
from pathlib import Path

import requests
from PIL import Image

REPO_ROOT = "/kaggle/working/yolov8s_kuangjing"
os.environ["PYTHONPATH"] = REPO_ROOT + ":" + os.environ.get("PYTHONPATH", "")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

!python -m pip install -q --no-deps -e /kaggle/working/yolov8s_kuangjing

assets_dir = Path(REPO_ROOT) / "ultralytics" / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)

    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / "bus.jpg", "https://ultralytics.com/images/bus.jpg")
ensure_valid_image(assets_dir / "zidane.jpg", "https://ultralytics.com/images/zidane.jpg")

!python -c "import ultralytics, sys; print('python=', sys.executable); print('ultralytics=', ultralytics.__file__)"


In [ ]:
# Cell 7: 公共训练参数
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_DEVICE = '0,1'   # 如果只有 1 张卡就改成 '0'
EPOCHS = 300
BATCH = 72            # 如果 OOM 就调小，比如 48 / 32
IMGSZ = 640
WORKERS = 6
RUN_PROJECT = '/kaggle/working/runs'
VAL_BATCH = 64
TEST_BATCH = 32

LSKA_RUN_NAME = 'yolov8s_lska_300ep'
INNER_IOU_RUN_NAME = 'yolov8s_inner_iou_300ep'
INNER_RATIO = 0.8

print('TRAIN_DEVICE =', TRAIN_DEVICE)
print('EPOCHS =', EPOCHS)
print('BATCH =', BATCH)
print('LSKA_RUN_NAME =', LSKA_RUN_NAME)
print('INNER_IOU_RUN_NAME =', INNER_IOU_RUN_NAME)
print('INNER_RATIO =', INNER_RATIO)


In [ ]:
# Cell 8: 训练 YOLOv8s + LSKA
# 只跑这一格时，就是 LSKA 实验
%cd /kaggle/working/yolov8s_kuangjing

import os
from ultralytics import YOLO

os.environ['ULTRALYTICS_CLS_LOSS'] = 'bce'
os.environ['ULTRALYTICS_IOU_LOSS'] = 'ciou'
os.environ.pop('ULTRALYTICS_INNER_IOU_RATIO', None)

ACTIVE_RUN_NAME = LSKA_RUN_NAME
ACTIVE_EXPERIMENT = 'lska'

model = YOLO('ultralytics/cfg/models/v8/yolov8s_lska.yaml')
model.train(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    task='detect',
    project=RUN_PROJECT,
    name=ACTIVE_RUN_NAME,
    device=TRAIN_DEVICE,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    pretrained='yolov8s.pt',
    save=True,
    val=True,
    plots=True,
    verbose=True,
    exist_ok=False,
)


In [ ]:
# Cell 9: 训练 YOLOv8s + Inner-IoU
# 只跑这一格时，就是 Inner-IoU 实验
%cd /kaggle/working/yolov8s_kuangjing

import os
from ultralytics import YOLO

os.environ['ULTRALYTICS_CLS_LOSS'] = 'bce'
os.environ['ULTRALYTICS_IOU_LOSS'] = 'inner_iou'
os.environ['ULTRALYTICS_INNER_IOU_RATIO'] = str(INNER_RATIO)

ACTIVE_RUN_NAME = INNER_IOU_RUN_NAME
ACTIVE_EXPERIMENT = 'inner_iou'

model = YOLO('yolov8s.pt')
model.train(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    task='detect',
    project=RUN_PROJECT,
    name=ACTIVE_RUN_NAME,
    device=TRAIN_DEVICE,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    workers=WORKERS,
    seed=SEED,
    deterministic=True,
    pretrained=True,
    save=True,
    val=True,
    plots=True,
    verbose=True,
    exist_ok=False,
)


In [ ]:
# Cell 10: 用 best.pt 分别在 val / test 上重新评估并保存结果
%cd /kaggle/working/yolov8s_kuangjing

from ultralytics import YOLO

assert 'ACTIVE_RUN_NAME' in globals(), '请先运行 Cell 8 或 Cell 9 中的一个训练单元。'
BEST = f'/kaggle/working/runs/{ACTIVE_RUN_NAME}/weights/best.pt'
print('BEST =', BEST)

model = YOLO(BEST)

print('===== VALIDATE ON VAL =====')
model.val(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='val',
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=0,
    plots=True,
    project='/kaggle/working/evals',
    name=f'{ACTIVE_RUN_NAME}_val',
)

print('===== VALIDATE ON TEST =====')
model.val(
    data='ultralytics/cfg/datasets/coalmine4_kaggle.yaml',
    split='test',
    imgsz=IMGSZ,
    batch=TEST_BATCH,
    device=0,
    plots=True,
    project='/kaggle/working/evals',
    name=f'{ACTIVE_RUN_NAME}_test',
)


In [ ]:
# Cell 11: 整理误差库（test 集预测可视化 + GT 标签 + 预测标签）
%cd /kaggle/working/yolov8s_kuangjing

from pathlib import Path
import shutil
from ultralytics import YOLO

assert 'ACTIVE_RUN_NAME' in globals(), '请先运行 Cell 8 或 Cell 9 中的一个训练单元。'
BEST = Path(f'/kaggle/working/runs/{ACTIVE_RUN_NAME}/weights/best.pt')
ERROR_ROOT = Path('/kaggle/working/error_bank') / ACTIVE_RUN_NAME
GT_LABELS = Path(DATASET_ROOT) / 'labels' / 'test'
TEST_IMAGES = Path(DATASET_ROOT) / 'images' / 'test'

if ERROR_ROOT.exists():
    shutil.rmtree(ERROR_ROOT)
ERROR_ROOT.mkdir(parents=True, exist_ok=True)

model = YOLO(str(BEST))
model.predict(
    source=str(TEST_IMAGES),
    imgsz=IMGSZ,
    conf=0.25,
    device=0,
    project=str(ERROR_ROOT.parent),
    name=ACTIVE_RUN_NAME,
    save=True,
    save_txt=True,
    save_conf=True,
)

GT_DST = ERROR_ROOT / 'test_gt_labels'
GT_DST.mkdir(parents=True, exist_ok=True)
for p in GT_LABELS.glob('*.txt'):
    shutil.copy2(p, GT_DST / p.name)

README = ERROR_ROOT / 'README.txt'
README.write_text(
    'error_bank contents:\n'
    '- 预测结果目录本身保存了 test 集可视化和 pred labels\n'
    '- test_gt_labels/: test 集对应的 GT labels\n'
    '- 结合 evals/ 下的 confusion_matrix 和 PR 曲线一起看误差\n',
    encoding='utf-8'
)
print('Error bank saved to', ERROR_ROOT)


In [ ]:
# Cell 12: 查看核心输出文件
assert 'ACTIVE_RUN_NAME' in globals(), '请先运行 Cell 8 或 Cell 9 中的一个训练单元。'
!ls /kaggle/working/runs/{ACTIVE_RUN_NAME}
!ls /kaggle/working/runs/{ACTIVE_RUN_NAME}/weights
!ls /kaggle/working/evals/{ACTIVE_RUN_NAME}_val || true
!ls /kaggle/working/evals/{ACTIVE_RUN_NAME}_test || true
!ls /kaggle/working/error_bank/{ACTIVE_RUN_NAME} || true


In [ ]:
# Cell 13: 打包训练结果、评估结果和误差库，方便下载
assert 'ACTIVE_RUN_NAME' in globals(), '请先运行 Cell 8 或 Cell 9 中的一个训练单元。'
!zip -r /kaggle/working/{ACTIVE_RUN_NAME}.zip /kaggle/working/runs/{ACTIVE_RUN_NAME}
!zip -r /kaggle/working/{ACTIVE_RUN_NAME}_evals.zip /kaggle/working/evals/{ACTIVE_RUN_NAME}_val /kaggle/working/evals/{ACTIVE_RUN_NAME}_test
!zip -r /kaggle/working/{ACTIVE_RUN_NAME}_error_bank.zip /kaggle/working/error_bank/{ACTIVE_RUN_NAME}
